In [3]:
#!/usr/bin/env python3
"""
S3 Large File Uploader - Notebook Compatible Version
Run this directly in Jupyter/IPython notebooks
"""

import logging
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import boto3
from botocore.config import Config
from botocore.exceptions import (
    BotoCoreError,
    ClientError,
    ReadTimeoutError,
    ConnectTimeoutError,
)

# Setup logging for notebooks
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)8s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True  # Override any existing config
)
logger = logging.getLogger(__name__)


class LargeMultipartUploader:
    """Upload a large file using robust multipart uploads."""

    def __init__(
        self,
        *,
        file_path: str,
        bucket: str,
        key: str,
        region: str,
        access_key: str,
        secret_key: str,
        endpoint: str,
        part_size: int = 64 * 1024 * 1024,
        max_retries: int = 10,
    ) -> None:
        self.file_path = file_path
        self.bucket = bucket
        self.key = key
        self.region = region
        self.access_key = access_key
        self.secret_key = secret_key
        self.endpoint = endpoint
        self.part_size = part_size
        self.max_retries = max_retries

        self.progress_lock = Lock()
        self.parts_completed = 0

        self.session = boto3.session.Session(
            aws_access_key_id=self.access_key,
            aws_secret_access_key=self.secret_key,
            region_name=self.region,
        )
        self.botocore_cfg = Config(
            region_name=self.region,
            retries={"max_attempts": self.max_retries, "mode": "standard"},
        )
        self.s3 = self.session.client(
            "s3", config=self.botocore_cfg, endpoint_url=self.endpoint
        )
        self.upload_id: str | None = None

    @staticmethod
    def human_mb_per_s(num_bytes: int, seconds: float) -> float:
        """Return MB/s as float, avoiding divide-by-zero."""
        return (num_bytes / (1024 * 1024)) / seconds if seconds > 0 else float("inf")

    @staticmethod
    def is_insufficient_storage_error(exc: Exception) -> bool:
        """Return True if the exception wraps a 507 Insufficient Storage response."""
        if isinstance(exc, ClientError):
            meta = exc.response.get("ResponseMetadata", {})
            return meta.get("HTTPStatusCode") == 507
        return False

    @staticmethod
    def is_524_error(exc: Exception) -> bool:
        """Return True if the exception wraps a 524 timeout response."""
        if isinstance(exc, ClientError):
            meta = exc.response.get("ResponseMetadata", {})
            return meta.get("HTTPStatusCode") == 524
        return False

    @staticmethod
    def is_520_error(exc: Exception) -> bool:
        """Return True if the exception wraps a 520 error response."""
        if isinstance(exc, ClientError):
            meta = exc.response.get("ResponseMetadata", {})
            return meta.get("HTTPStatusCode") == 520
        return False

    @staticmethod
    def is_no_such_upload_error(exc: Exception) -> bool:
        """Return True if the exception reports a missing multipart upload."""
        if isinstance(exc, ClientError):
            err = exc.response.get("Error", {})
            return err.get("Code") == "NoSuchUpload"
        return False

    def call_with_524_retry(self, description: str, func):
        """Call ``func`` retrying on HTTP 524/520 or timeout errors."""
        for attempt in range(1, self.max_retries + 1):
            try:
                return func()
            except ClientError as exc:
                if self.is_524_error(exc) or self.is_520_error(exc):
                    status = "524" if self.is_524_error(exc) else "520"
                    logger.warning(
                        f"{description}: received {status} response (attempt {attempt})"
                    )
                    if attempt == self.max_retries:
                        logger.error(f"{description}: exceeded max_retries for {status}")
                        raise
                    backoff = 2**attempt
                    logger.info(f"{description}: retrying in {backoff}s...")
                    time.sleep(backoff)
                    continue
                raise
            except (ReadTimeoutError, ConnectTimeoutError) as exc:
                logger.warning(
                    f"{description}: request timed out (attempt {attempt}): {exc}"
                )
                if attempt == self.max_retries:
                    logger.error(f"{description}: exceeded max_retries for timeout")
                    raise
                backoff = 2**attempt
                logger.info(f"{description}: retrying in {backoff}s...")
                time.sleep(backoff)

    def complete_with_timeout_retry(
        self,
        *,
        parts_sorted: list,
        initial_timeout: int,
        expected_size: int,
    ):
        """Complete the multipart upload, doubling timeout on client timeouts."""
        if self.upload_id is None:
            raise RuntimeError("upload_id not set")

        timeout = initial_timeout
        cfg = self.botocore_cfg
        last_exc: Exception | None = None
        for attempt in range(1, self.max_retries + 1):
            cfg = cfg.merge(Config(read_timeout=timeout, connect_timeout=timeout))
            client = self.session.client("s3", config=cfg, endpoint_url=self.endpoint)
            try:
                client.complete_multipart_upload(
                    Bucket=self.bucket,
                    Key=self.key,
                    UploadId=self.upload_id,
                    MultipartUpload={"Parts": parts_sorted},
                )
                self.s3 = client
                self.botocore_cfg = cfg
                return
            except (ReadTimeoutError, ConnectTimeoutError) as exc:
                last_exc = exc
                no_such_upload = False
                logger.warning(
                    f"complete_multipart_upload timed out after {timeout}s: {exc}"
                )
            except (ClientError, BotoCoreError) as exc:
                last_exc = exc
                no_such_upload = self.is_no_such_upload_error(exc)
                logger.warning(
                    f"complete_multipart_upload failed (attempt {attempt}): {exc}"
                )

            if no_such_upload:
                logger.info("Upload session missing; checking object state immediately")
            else:
                logger.info(
                    f"Waiting {timeout}s before checking object state to see if merge has completed"
                )
                time.sleep(timeout)

            try:
                head = self.call_with_524_retry(
                    "head_object",
                    lambda: client.head_object(Bucket=self.bucket, Key=self.key),
                )
                uploaded_size = head.get("ContentLength")
                if uploaded_size == expected_size:
                    logger.info(
                        "HeadObject confirms multipart upload merge has completed"
                    )
                    self.s3 = client
                    self.botocore_cfg = cfg
                    return
                logger.info(
                    "HeadObject size mismatch after timeout; will retry complete_multipart_upload"
                )
            except Exception as head_exc:
                logger.info(f"head_object failed after error: {head_exc}")

            if attempt == self.max_retries:
                raise (
                    last_exc
                    if last_exc
                    else RuntimeError(
                        "Exceeded max_retries without completing multipart upload"
                    )
                )

            timeout *= 2
            logger.info(f"Increasing timeout to {timeout}s and retrying")

    def upload_part(
        self,
        *,
        part_number: int,
        offset: int,
        bytes_to_read: int,
        total_parts: int,
        start_time: float,
    ) -> dict:
        """Upload a single part with exponential-backoff retries."""
        if self.upload_id is None:
            raise RuntimeError("upload_id not set")

        for attempt in range(1, self.max_retries + 1):
            try:
                logger.info(
                    f"Part {part_number}: reading bytes {offset}–{offset+bytes_to_read} (attempt {attempt})"
                )
                with open(self.file_path, "rb") as f:
                    f.seek(offset)
                    data = f.read(bytes_to_read)
                resp = self.s3.upload_part(
                    Bucket=self.bucket,
                    Key=self.key,
                    PartNumber=part_number,
                    UploadId=self.upload_id,
                    Body=data,
                )
                etag = resp["ETag"]
                with self.progress_lock:
                    self.parts_completed += 1
                    progress = 100.0 * self.parts_completed / total_parts
                elapsed = time.time() - start_time
                progress_fraction = part_number / total_parts
                if progress_fraction > 0:
                    remaining = max(0, elapsed * (1 / progress_fraction - 1))
                    eta = time.strftime("%Hh %Mm %Ss", time.gmtime(remaining))
                else:
                    eta = "?"
                logger.info(
                    f"Part {part_number}: uploaded, progress: {progress:.1f}%, est time remaining: {eta}"
                )
                return {"PartNumber": part_number, "ETag": etag}
            except (BotoCoreError, ClientError) as exc:
                if self.is_insufficient_storage_error(exc):
                    logger.error(
                        f"Part {part_number}: received 507 Insufficient Storage; aborting"
                    )
                    raise RuntimeError("Server reported insufficient storage") from exc
                if self.is_524_error(exc) or self.is_520_error(exc):
                    status = "524" if self.is_524_error(exc) else "520"
                    logger.warning(
                        f"Part {part_number}: received {status} response (attempt {attempt})"
                    )
                else:
                    logger.warning(
                        f"Part {part_number}: attempt {attempt} failed: {exc}"
                    )
                if attempt == self.max_retries:
                    logger.error(
                        f"Part {part_number}: exceeded max_retries ({self.max_retries})"
                    )
                    raise
                backoff = 2**attempt
                logger.info(f"Part {part_number}: retrying in {backoff}s...")
                time.sleep(backoff)

    def upload(self) -> None:
        """Execute the multipart upload."""
        logger.info(
            f"Uploading to region: {self.region}; bucket: {self.bucket}; key: {self.key}"
        )

        file_size = os.path.getsize(self.file_path)
        total_parts = math.ceil(file_size / self.part_size)
        logger.info(
            f"File size: {file_size} bytes ({file_size / (1024**3):.2f} GB); "
            f"will upload in {total_parts} parts of up to {self.part_size} bytes each"
        )

        start_time = time.time()

        file_gb = file_size / float(1024**3)
        completion_timeout = max(60, int(math.ceil(file_gb) * 5))

        resp = self.call_with_524_retry(
            "create_multipart_upload",
            lambda: self.s3.create_multipart_upload(Bucket=self.bucket, Key=self.key),
        )
        self.upload_id = resp["UploadId"]
        logger.info(f"Initiated multipart upload: UploadId={self.upload_id}")

        parts: list[dict] = []
        try:
            with ThreadPoolExecutor(max_workers=4) as executor:
                futures = {}
                for part_num in range(1, total_parts + 1):
                    offset = (part_num - 1) * self.part_size
                    chunk_size = min(self.part_size, file_size - offset)
                    futures[
                        executor.submit(
                            self.upload_part,
                            part_number=part_num,
                            offset=offset,
                            bytes_to_read=chunk_size,
                            total_parts=total_parts,
                            start_time=start_time,
                        )
                    ] = part_num

                for fut in as_completed(futures):
                    part = fut.result()
                    parts.append(part)

            def fetch_parts():
                paginator = self.s3.get_paginator("list_parts")
                found = []
                for page in paginator.paginate(
                    Bucket=self.bucket, Key=self.key, UploadId=self.upload_id
                ):
                    found.extend(page.get("Parts", []))
                return found

            seen = self.call_with_524_retry("list_parts", fetch_parts)
            logger.info(f"Verified {len(seen)} of {total_parts} parts uploaded")

            if len(seen) != total_parts:
                raise RuntimeError(f"Expected {total_parts} parts but saw {len(seen)}")

            parts_sorted = sorted(parts, key=lambda x: x["PartNumber"])
            logger.info("Sending complete_multipart_upload request")
            self.complete_with_timeout_retry(
                parts_sorted=parts_sorted,
                initial_timeout=completion_timeout,
                expected_size=file_size,
            )

            head = self.call_with_524_retry(
                "head_object",
                lambda: self.s3.head_object(Bucket=self.bucket, Key=self.key),
            )
            uploaded_size = head.get("ContentLength")
            if uploaded_size != file_size:
                logger.error(
                    f"Size mismatch: remote object is {uploaded_size} bytes, "
                    f"but local file is {file_size} bytes"
                )
                raise RuntimeError(
                    "Multipart upload verification failed: size mismatch"
                )
            logger.info(
                f"Verified upload: remote object size {uploaded_size} bytes matches local file size"
            )
        except Exception as exc:
            logger.error(f"Upload interrupted: {exc}")
            if self.upload_id:
                logger.info(f"UploadId {self.upload_id} left open for resumption")
            raise

        elapsed = time.time() - start_time
        speed = self.human_mb_per_s(file_size, elapsed)
        duration = time.strftime("%Hh %Mm %Ss", time.gmtime(elapsed))
        logger.info(f"✓ Upload completed! Speed {speed:.2f} MB/s, Duration {duration}")


# =============================================================================
# NOTEBOOK USAGE EXAMPLE
# =============================================================================

def upload_file(
    file_path: str,
    bucket: str,
    key: str,
    region: str = "us-il-1",
    endpoint: str = "https://s3api-us-il-1.runpod.io/",
    access_key: str = None,
    secret_key: str = None,
    chunk_size: int = 64 * 1024 * 1024,
    max_retries: int = 10,
):
    """
    Convenience function for notebook usage.
    
    Example:
        upload_file(
            file_path="~/ComfyUI/models/diffusion_models/Chroma1-HD-fp8_scaled_rev2.safetensors",
            bucket="z6y2yim0s3",
            key="models/diffusion_models/Chroma1-HD-fp8_scaled_rev2.safetensors"
        )
    """
    # Expand user path
    file_path = os.path.expanduser(file_path)
    
    # Get credentials from environment if not provided
    if access_key is None:
        access_key = os.environ.get("AWS_ACCESS_KEY_ID")
    if secret_key is None:
        secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
    
    if not access_key or not secret_key:
        raise ValueError("AWS credentials not found. Set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY")
    
    uploader = LargeMultipartUploader(
        file_path=file_path,
        bucket=bucket,
        key=key,
        region=region,
        access_key=access_key,
        secret_key=secret_key,
        endpoint=endpoint,
        part_size=chunk_size,
        max_retries=max_retries,
    )
    {
  "cells": [
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "#!/usr/bin/env python3\n",
        "\"\"\"\n",
        "Multipart uploader for very large files.\n",
        "Notebook-friendly version using argparse(parse_args(args=[])).\n",
        "\"\"\"\n",
        "\n",
        "import argparse\n",
        "import logging\n",
        "import math\n",
        "import os\n",
        "import sys\n",
        "import time\n",
        "from concurrent.futures import ThreadPoolExecutor, as_completed\n",
        "from threading import Lock\n",
        "\n",
        "import boto3\n",
        "from botocore.config import Config\n",
        "from botocore.exceptions import (\n",
        "    BotoCoreError,\n",
        "    ClientError,\n",
        "    ReadTimeoutError,\n",
        "    ConnectTimeoutError,\n",
        ")\n",
        "\n",
        "def parse_args(args=None):\n",
        "    parser = argparse.ArgumentParser(description=\"Multipart uploader\")\n",
        "    parser.add_argument(\"-b\", \"--bucket\", required=True)\n",
        "    parser.add_argument(\"-f\", \"--file\", dest=\"file_path\", required=True)\n",
        "    parser.add_argument(\"-k\", \"--key\", required=True)\n",
        "    parser.add_argument(\"-c\", \"--chunk-size\", type=int, default=50 * 1024 * 1024)\n",
        "    parser.add_argument(\"-a\", \"--access_key\", default=os.environ.get(\"AWS_ACCESS_KEY_ID\"))\n",
        "    parser.add_argument(\"-s\", \"--secret_key\", default=os.environ.get(\"AWS_SECRET_ACCESS_KEY\"))\n",
        "    parser.add_argument(\"-e\", \"--endpoint\", default=os.environ.get(\"S3_ENDPOINT\"))\n",
        "    parser.add_argument(\"-r\", \"--region\", default=os.environ.get(\"S3_REGION\", \"us-east-1\"))\n",
        "    parser.add_argument(\"-m\", \"--max-retries\", type=int, default=5)\n",
        "    return parser.parse_args(args=args)\n",
        "\n",
        "logging.basicConfig(level=logging.INFO, format=\"[%(asctime)s] %(levelname)s %(message)s\")\n",
        "logger = logging.getLogger(__name__)\n",
        "\n",
        "class LargeMultipartUploader:\n",
        "    def __init__(self, *, file_path, bucket, key, region, access_key, secret_key, endpoint, part_size=50*1024*1024, max_retries=5):\n",
        "        self.file_path = file_path\n",
        "        self.bucket = bucket\n",
        "        self.key = key\n",
        "        self.region = region\n",
        "        self.access_key = access_key\n",
        "        self.secret_key = secret_key\n",
        "        self.endpoint = endpoint\n",
        "        self.part_size = part_size\n",
        "        self.max_retries = max_retries\n",
        "        self.progress_lock = Lock()\n",
        "        self.parts_completed = 0\n",
        "        self.session = boto3.session.Session(\n",
        "            aws_access_key_id=self.access_key,\n",
        "            aws_secret_access_key=self.secret_key,\n",
        "            region_name=self.region,\n",
        "        )\n",
        "        self.botocore_cfg = Config(region_name=self.region, retries={\"max_attempts\": self.max_retries, \"mode\": \"standard\"})\n",
        "        self.s3 = self.session.client(\"s3\", config=self.botocore_cfg, endpoint_url=self.endpoint)\n",
        "        self.upload_id = None\n",
        "\n",
        "    def call_with_retry(self, desc, func):\n",
        "        for attempt in range(1, self.max_retries+1):\n",
        "            try:\n",
        "                return func()\n",
        "            except Exception as exc:\n",
        "                if attempt == self.max_retries:\n",
        "                    raise\n",
        "                time.sleep(2**attempt)\n",
        "\n",
        "    def upload_part(self, *, part_number, offset, bytes_to_read):\n",
        "        with open(self.file_path, \"rb\") as f:\n",
        "            f.seek(offset)\n",
        "            data = f.read(bytes_to_read)\n",
        "        resp = self.s3.upload_part(Bucket=self.bucket, Key=self.key, PartNumber=part_number, UploadId=self.upload_id, Body=data)\n",
        "        return {\"PartNumber\": part_number, \"ETag\": resp[\"ETag\"]}\n",
        "\n",
        "    def upload(self):\n",
        "        file_size = os.path.getsize(self.file_path)\n",
        "        total_parts = math.ceil(file_size / self.part_size)\n",
        "        self.upload_id = self.call_with_retry(\"create_multipart_upload\", lambda: self.s3.create_multipart_upload(Bucket=self.bucket, Key=self.key))[\"UploadId\"]\n",
        "        parts = []\n",
        "        with ThreadPoolExecutor(max_workers=4) as exe:\n",
        "            futures = []\n",
        "            for pn in range(1, total_parts+1):\n",
        "                offset = (pn-1) * self.part_size\n",
        "                size = min(self.part_size, file_size-offset)\n",
        "                futures.append(exe.submit(self.upload_part, part_number=pn, offset=offset, bytes_to_read=size))\n",
        "            for f in as_completed(futures): parts.append(f.result())\n",
        "\n",
        "        parts_sorted = sorted(parts, key=lambda x: x[\"PartNumber\"])\n",
        "        self.call_with_retry(\"complete_multipart_upload\", lambda: self.s3.complete_multipart_upload(\n",
        "            Bucket=self.bucket,\n",
        "            Key=self.key,\n",
        "            UploadId=self.upload_id,\n",
        "            MultipartUpload={\"Parts\": parts_sorted},\n",
        "        ))\n",
        "        print(\"Upload complete!\")\n",
        "\n",
        "# Notebook config section\n",
        "BUCKET = \"\"\n",
        "FILE_PATH = \"\"\n",
        "KEY = \"\"\n",
        "REGION = os.environ.get(\"S3_REGION\", \"us-east-1\")\n",
        "ENDPOINT = os.environ.get(\"S3_ENDPOINT\", \"\")\n",
        "ACCESS_KEY = os.environ.get(\"AWS_ACCESS_KEY_ID\", \"\")\n",
        "SECRET_KEY = os.environ.get(\"AWS_SECRET_ACCESS_KEY\", \"\")\n",
        "CHUNK_SIZE = 50 * 1024 * 1024\n",
        "\n",
        "if all([BUCKET, FILE_PATH, KEY, ENDPOINT, ACCESS_KEY, SECRET_KEY]):\n",
        "    uploader = LargeMultipartUploader(\n",
        "        file_path=FILE_PATH,\n",
        "        bucket=BUCKET,\n",
        "        key=KEY,\n",
        "        region=REGION,\n",
        "        access_key=ACCESS_KEY,\n",
        "        secret_key=SECRET_KEY,\n",
        "        endpoint=ENDPOINT,\n",
        "        part_size=CHUNK_SIZE,\n",
        "    )\n",
        "    # uploader.upload()   # Uncomment to run\n",
        "else:\n",
        "    print(\"\u26a0\ufe0f Fill in BUCKET, FILE_PATH, KEY, ENDPOINT, ACCESS_KEY, SECRET_KEY.\")\n"
      ],
      "execution_count": null,
      "outputs": []
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.10"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
    uploader.upload()


# Example usage in notebook:
if __name__ == "__main__":
    # Uncomment and modify to use:
    """
    upload_file(
        file_path="~/ComfyUI/models/diffusion_models/Chroma1-HD-fp8_scaled_rev2.safetensors",
        bucket="z6y2yim0s3",
        key="models/diffusion_models/Chroma1-HD-fp8_scaled_rev2.safetensors",
        chunk_size=64 * 1024 * 1024,  # 64 MiB
    )
    """
    print("Import this module and call upload_file() with your parameters")

Import this module and call upload_file() with your parameters
